# Circuit Breakers — Mechanistic Explanation via Displacement (E5)

Tests whether the per-turn displacement of `v_harmful` geometrically explains the known failure of circuit breakers against multi-turn jailbreaks.

Plan: `docs/circuit_breakers_plan.md`. Hypotheses:

| # | Claim | Tier |
|---|-------|------|
| **H-CB1** | Rerouting direction `v_rr` aligns with `v_harmful_base(ST)` but becomes orthogonal to `v_harmful_base(k)` at later k | **Headline** |
| **H-CB2** | Displacement persists on the CB model — `cos(v_full(k), v_ST)` drops with k, same shape as base | Primary |
| **H-CB3** | Per-attempt displacement magnitude correlates with jailbreak success on the CB model | Supplement |
| **H-CB4** | CB training drops single-turn ASR sharply; leaves multi-turn ASR largely intact | Primary |

**Required data (not yet generated):**
- `data/llama_3_base/` — conversations, trajectories, nocontext, compressed, single-turn on Meta-Llama-3-8B-Instruct
- `data/llama_3_RR/`   — same on GraySwanAI/Llama-3-8B-Instruct-RR
- Both scored under AISI-StrongREJECT via nb02

Until the data lands, every analysis cell below is a placeholder that raises `NotImplementedError` with a clear next step.


## 1. Setup

In [1]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import roc_auc_score

repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

BASE_MODEL = 'llama_3_base'     # under data/
RR_MODEL   = 'llama_3_RR'       # under data/
FRAMEWORKS = ['crescendo', 'actorattack', 'xteaming']
SPLITS     = ['harmful', 'benign']
H_KEY      = 'h_inst'
MAX_K      = 10
RNG_SEED   = 42

DATA_ROOT = repo_root / 'data'
FIG_DIR   = repo_root / 'figures'
FIG_DIR.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})

# HuggingFace model IDs (for Option-A weight-diff analysis in §3)
BASE_HF = 'meta-llama/Meta-Llama-3-8B-Instruct'
RR_HF   = 'GraySwanAI/Llama-3-8B-Instruct-RR'

print(f'BASE_MODEL={BASE_MODEL}   RR_MODEL={RR_MODEL}')


BASE_MODEL=llama_3_base   RR_MODEL=llama_3_RR


In [2]:
# Shared loader — mirrors nb08 load_model()
from datasets import load_dataset

TRAIN_MAX_ATTEMPT = 16

def _load_layer_info(model_root):
    for sub in ('trajectories', 'nocontext', 'single_turn'):
        root = model_root / sub
        if not root.exists():
            continue
        for d in root.iterdir():
            fp = d / 'layer_indices.json'
            if fp.exists():
                return json.loads(fp.read_text())
    return None

def load_model(name):
    repr_root = DATA_ROOT / name / 'representations'
    if not repr_root.exists():
        return None
    layer_info = _load_layer_info(repr_root)
    if layer_info is None:
        return None
    out = {'name': name, 'repr_root': repr_root, 'layer_info': layer_info,
           'traj': {}, 'nc': {}, 'comp': {}, 'st': {}}
    for fw in FRAMEWORKS:
        for split in SPLITS:
            for key, sub in [('traj','trajectories'), ('nc','nocontext'), ('comp','compressed')]:
                folder = repr_root / sub / f'{fw}_{split}'
                if folder.exists():
                    meta = pd.read_parquet(folder / 'metadata.parquet')
                    meta['data_split'] = np.where(meta['attempt'] <= TRAIN_MAX_ATTEMPT, 'train', 'test')
                    out[key][(fw, split)] = {
                        'meta':   meta,
                        'h_inst': np.load(folder / 'h_inst.npy',       mmap_mode='r'),
                        'h_post': np.load(folder / 'h_post_inst.npy',  mmap_mode='r'),
                    }
    for split in SPLITS:
        folder = repr_root / 'single_turn' / split
        if folder.exists():
            out['st'][split] = {
                'meta':   pd.read_parquet(folder / 'metadata.parquet'),
                'h_inst': np.load(folder / 'h_inst.npy',       mmap_mode='r'),
                'h_post': np.load(folder / 'h_post_inst.npy',  mmap_mode='r'),
            }
    return out

BASE = load_model(BASE_MODEL)
RR   = load_model(RR_MODEL)

if BASE is None or RR is None:
    print('[scaffold] base or RR representations missing. Expected:')
    print(f'   {DATA_ROOT / BASE_MODEL / "representations"}')
    print(f'   {DATA_ROOT / RR_MODEL   / "representations"}')
    print('Until present, analysis cells below raise NotImplementedError.')
else:
    print(f'[loaded] base: {sum(len(v["meta"]) for v in BASE["traj"].values()):6d} rows')
    print(f'[loaded] RR:   {sum(len(v["meta"]) for v in RR["traj"].values()):6d} rows')


/Users/lisahusieva/Desktop/multi-turn-rep-eng/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[scaffold] base or RR representations missing. Expected:
   /Users/lisahusieva/Desktop/multi-turn-rep-eng/data/llama_3_base/representations
   /Users/lisahusieva/Desktop/multi-turn-rep-eng/data/llama_3_RR/representations
Until present, analysis cells below raise NotImplementedError.


## 2. Direction computation on base and CB models

Same paired-subtraction pipeline as nb03/nb04 (within-pair mean-difference, normalized). Computed at each saved layer under each extraction condition.

`v_harmful_base(ST)`, `v_harmful_base(k)` for each k, and same for RR.


In [ ]:
def _raise_if_missing():
    if BASE is None or RR is None:
        raise NotImplementedError(
            'Generate data/llama_3_base and data/llama_3_RR before running this cell. '
            'See docs/circuit_breakers_plan.md §4.1.'
        )

def compute_direction_paired(h_harm, m_harm, h_beni, m_beni):
    pairs = sorted(set(m_harm['pair_id']) & set(m_beni['pair_id']))
    diffs = []
    for pid in pairs:
        idx_h = m_harm.index[m_harm['pair_id'] == pid].tolist()
        idx_b = m_beni.index[m_beni['pair_id'] == pid].tolist()
        if not idx_h or not idx_b:
            continue
        mu_h = h_harm[idx_h].astype(np.float32).mean(0)
        mu_b = h_beni[idx_b].astype(np.float32).mean(0)
        diffs.append(mu_h - mu_b)
    if not diffs:
        return None
    d = np.stack(diffs).mean(0)
    return d / (np.linalg.norm(d, axis=-1, keepdims=True) + 1e-12)

# TODO: replicate nb08's v_st / v_full(k) / v_nc(k) / v_comp computation for both BASE and RR.
_raise_if_missing()


## 3. Locating the rerouting direction `v_rr`

Two estimates, per intervened layer:

**Option A — weight-difference.** Let `ΔW_l = W_l^RR − W_l^base` at the intervened layers. Top singular vector of `ΔW_l` approximates the rerouting direction.

**Option B — activation-difference.** On paired (harmful, benign) JBB goals, compute `h_l^RR − h_l^base`. Mean across examples gives the empirical rerouting direction.

Use both; they should agree up to sign and magnitude. Report as `v_rr_l` per intervened layer, and the top-K basis `V_rr_l^{1..K}` since the intervention may span more than rank-1.

**Intervention layers not yet known** — identify from `||ΔW_l||_F` peaks (largest-norm layers = intervention targets).


In [ ]:
def weight_diff_direction(layer, top_k=5):
    """Load base and RR weights at `layer` (via HF), compute top-K singular vectors of ΔW."""
    # Implementation deferred — requires loading both models' weights (~16GB x 2).
    # Uses transformers.AutoModelForCausalLM.from_pretrained(..., torch_dtype=torch.bfloat16).
    # Extract residual-stream-relevant weights (likely attn.o_proj or mlp.down_proj output).
    raise NotImplementedError('Implement when base+RR HF weights are available locally.')

def activation_diff_direction(layer):
    """Empirical rerouting: h_l^RR − h_l^base on paired JBB single-turn reps."""
    _raise_if_missing()
    # Pair h_inst from BASE['st']['harmful'] with RR['st']['harmful'] by pair_id.
    # Compute mean diff per pair, then grand mean, normalize.
    raise NotImplementedError('Implement after extraction runs on both models.')

def intervention_layers_from_delta_w(threshold_percentile=90):
    """Layers with top-decile ||ΔW_l||_F are the intervention targets."""
    raise NotImplementedError('Implement after weight_diff_direction is available.')


## 4. H-CB1 — Rerouting direction vs `v_harmful_base(k)`

For each intervened layer `l`, plot `cos(v_rr_l, v_harmful_base(k)_l)` as a function of turn `k`, one panel per framework.

**Prediction (H-CB1):** descending line — CB's rerouting axis is close to the single-turn anchor but orthogonal to late-turn directions.

This is the headline geometric claim of the paper's CB section.


In [ ]:
# Fig CB-1: rerouting × displacement cosine
_raise_if_missing()
# TODO: For each intervened layer l:
#   1. Compute v_rr_l via weight-diff (Option A) and activation-diff (Option B).
#   2. For each fw and k in 1..MAX_K, compute cos(v_rr_l, v_harmful_base_full(k)_l).
#   3. Plot as a line per framework, one subplot per intervened layer.
#   4. Save to FIG_DIR / '09_fig1_rr_vs_displacement.png'.
raise NotImplementedError('Headline CB figure — implement after directions + v_rr computed.')


## 5. H-CB2 — CB model's own displacement

Replicate nb04 Fig 1 with `MODEL=llama_3_RR`: plot `cos(v_full_RR(k), v_ST_RR)` and `cos(v_full_RR(k), v_nc_RR(k))` across k.

**Prediction (H-CB2):** same shape as base Llama-3 — CB training does not close the geometric gap that drives displacement.


In [ ]:
# Fig CB-2: CB model's own displacement (mirrors nb04 Fig 1)
_raise_if_missing()
# TODO: Compute per-turn directions on RR, plot cos vs v_ST_RR and v_nc_RR(k).
# Save to FIG_DIR / '09_fig2_rr_displacement.png'.
raise NotImplementedError('Displacement-on-RR figure — implement after RR directions computed.')


## 6. H-CB4 — Single-turn vs multi-turn ASR drop

Under AISI-StrongREJECT, compare base and CB ASR on:
- Single-turn JBB goals (no attack framing)
- Crescendo / ActorAttack / X-Teaming

**Prediction (H-CB4):** sharp drop only in the single-turn column; multi-turn stays close to base.

This directly quantifies the defense failure that H-CB1/H-CB2 explain mechanistically.


In [ ]:
# Fig CB-3: base vs RR ASR bar chart under AISI
# Load nb02 output: data/{model}/sr_final_scores.csv for both BASE and RR.
# Include a 'single-turn JBB' group — this requires running the single-turn baseline as
# a 1-turn 'conversation' under the same scoring pipeline. See circuit_breakers_plan.md §4.4.
_raise_if_missing()
raise NotImplementedError('ASR comparison — implement after AISI scoring on both models.')


## 7. H-CB3 — Per-attempt displacement × success (supplement)

On the CB model, correlate per-attempt displacement magnitude (e.g. `||h_k − h_1||` at mid layer, or `1 − cos(v_k, v_ST)` averaged over attempt turns) with the AISI-jailbroken binary label.

**Prediction (H-CB3):** monotonic positive relationship — attempts that displace more are more likely to succeed. Reported as supplement; main claim rests on H-CB1/H-CB2/H-CB4.


In [ ]:
# Fig CB-4 (supplement): binned displacement × jailbreak probability
_raise_if_missing()
# TODO: For each harmful RR conversation, compute per-attempt displacement scalar.
# Bin by decile, plot fraction jailbroken per bin.
# Save to FIG_DIR / '09_fig4_displacement_success.png'.
raise NotImplementedError('Per-attempt correlation — implement after directions + AISI labels on RR.')


## 8. Probe quality on the CB model

Re-run the E1 certifications (nb03 Checks 7–10) on the CB model's directions. If `v_harmful_RR(k)` is noisier or more topic-entangled than base, that's itself a finding — it bounds how strongly we can interpret H-CB2.

Not strictly a hypothesis test but a precondition for trusting the other four.


In [ ]:
# Delegate to nb03 by setting MODEL='llama_3_RR' at its top, or extract
# the four certification functions into src/probe_quality.py and call here.
_raise_if_missing()
raise NotImplementedError(
    'Run 03_probe_quality.ipynb with MODEL="llama_3_RR" — no new code needed here.'
)


## 9. Summary — what the figures should land

If H-CB1, H-CB2, H-CB4 all hold (and H-CB3 at least leans positive), the paper's claim becomes:

> **Circuit breakers reroute representations along a direction close to `v_harmful(ST)`; multi-turn attacks move `v_harmful(k)` off that direction, so the defense is pointed at a stale anchor by the time of jailbreak.** CB drops single-turn ASR substantially but leaves multi-turn ASR largely intact. The displacement phenomenon geometrically accounts for the known multi-turn failure of the canonical representation-level defense.

Upgrades the paper's contribution from *"displacement correlates with single-turn defenses degrading"* to *"displacement mechanistically explains the failure of the canonical representation-level defense."*

### Execution order (per experiment_plan.md §7)

1. Scaffold `notebooks/llama_3_8B_instruct/` + `notebooks/llama_3_8B_RR_instruct/` from the 3.1 template.
2. Smoke-test generation (3 pairs, 2 attempts, Crescendo only) on both models.
3. Full data gen + extraction on both models (Week 4).
4. Return here and flip the placeholders into implementations (Week 5).
